# 04 — File Tools：文件系统工具

工具框架有了，现在造**真正有用的工具**。

Coding Agent 最高频的操作就是跟文件打交道：读代码、写代码、找文件、改一行。这章实现 5 个文件系统工具：

| 工具 | 功能 |
|------|------|
| `ReadFileTool` | 读取文件内容，支持分页 |
| `WriteFileTool` | 写入/覆盖文件，生成 diff |
| `EditTool` | 精确替换文件中的某段文字 |
| `ListDirectoryTool` | 列出目录内容 |
| `GlobTool` | 按模式搜索文件（`**/*.py`） |

---

文件工具看起来简单，但细节很多——路径安全、大文件处理、二进制文件检测、diff 生成。每个都要处理好，不然 Agent 一跑就出问题。

## 学完这章你能做到

- 实现安全的路径解析（防止 Agent 读写工作目录之外的文件）
- 处理大文件（分页读取，避免塞满 token）
- 生成 unified diff，让 LLM 和用户都能看清楚改了什么
- 用 glob 模式高效定位文件

In [ ]:
import sys
sys.path.insert(0, "..")

from src.tool_framework import Tool, ToolKind, ToolResult, ToolRegistry

## 1. 路径安全：最重要的前置工作

Agent 会把 LLM 生成的路径直接传给文件操作。LLM 可能生成这样的路径：
- `../../etc/passwd`（路径穿越攻击）
- `/absolute/path/outside/workspace`（读写工作目录外的文件）

规则很简单：**文件操作只允许在工作目录及其子目录内**。

`resolve_path()` 负责把用户传入的路径规范化，并验证是否在工作目录内。

In [ ]:
from pathlib import Path


def resolve_path(path_str: str, cwd: Path) -> Path:
    """
    把字符串路径解析为绝对 Path，并验证在工作目录内。
    
    - 相对路径：相对于 cwd 解析
    - 绝对路径：直接使用，但仍然检查是否在 cwd 内
    - 路径穿越（../../）：通过 resolve() 展开后检查
    
    如果路径在 cwd 外，抛出 PermissionError。
    """
    p = Path(path_str)
    
    # 相对路径 → 基于 cwd 解析
    if not p.is_absolute():
        p = cwd / p
    
    # resolve() 展开所有 .. 和软链接，得到真实绝对路径
    # strict=False：路径不存在时也能解析（写新文件时需要）
    resolved = p.resolve()
    cwd_resolved = cwd.resolve()
    
    # 检查是否在工作目录内（is_relative_to 是 Python 3.9+）
    if not (resolved == cwd_resolved or str(resolved).startswith(str(cwd_resolved) + "/")):
        raise PermissionError(
            f"路径 '{path_str}' 在工作目录之外。"
            f"工作目录: {cwd_resolved}"
        )
    
    return resolved


# 测试
cwd = Path(".").resolve()
print(f"当前工作目录: {cwd}")
print()

# 正常路径
print("正常路径:")
print(" ", resolve_path("src/llm_client.py", cwd))
print(" ", resolve_path("./readme.md", cwd))
print()

# 路径穿越尝试
print("路径穿越尝试:")
try:
    resolve_path("../../etc/passwd", cwd)
except PermissionError as e:
    print(f"  被拦截: {e}")

## 2. 工具一：ReadFileTool

读文件看起来就是 `open().read()`，但有几个问题要处理：

- **大文件**：一个 10000 行的文件全读出来会消耗大量 token，要支持分页（`offset` / `limit`）
- **二进制文件**：图片、音频读出来是乱码，要检测并拒绝
- **行号**：返回带行号的内容，方便 LLM 告诉你「第 42 行有问题」
- **Token 截断**：即使分页了，内容可能还是太长，超过阈值要截断

In [ ]:
def is_binary(path: Path) -> bool:
    """通过读取文件开头来判断是否是二进制文件。"""
    try:
        with open(path, "rb") as f:
            chunk = f.read(8192)
        # 文本文件里不会有 null 字节
        return b"\x00" in chunk
    except Exception:
        return False


def add_line_numbers(text: str, start_line: int = 1) -> str:
    """给文本加行号，格式：'  42 │ content'"""
    lines = text.splitlines()
    width = len(str(start_line + len(lines) - 1))  # 行号位数
    result = []
    for i, line in enumerate(lines):
        lineno = start_line + i
        result.append(f"{lineno:>{width}} │ {line}")
    return "\n".join(result)


# 验证
sample = "def hello():\n    print('hello')\n    return True"
print(add_line_numbers(sample, start_line=10))

In [ ]:
MAX_FILE_SIZE = 10 * 1024 * 1024   # 10 MB，超过不读
MAX_READ_CHARS = 20_000            # 单次读取最多返回这么多字符给 LLM


class ReadFileTool(Tool):

    def __init__(self, cwd: Path = Path(".")):
        self.cwd = cwd.resolve()

    @property
    def name(self) -> str:
        return "read_file"

    @property
    def description(self) -> str:
        return (
            "读取文件内容。返回带行号的文本，便于定位代码位置。"
            "支持 offset/limit 分页读取大文件。"
            "不支持二进制文件（图片、音频等）。"
        )

    @property
    def kind(self) -> ToolKind:
        return ToolKind.READ

    def schema(self) -> dict:
        return {
            "type": "function",
            "function": {
                "name": self.name,
                "description": self.description,
                "parameters": {
                    "type": "object",
                    "properties": {
                        "path": {
                            "type": "string",
                            "description": "文件路径（相对于工作目录或绝对路径）"
                        },
                        "offset": {
                            "type": "integer",
                            "description": "从第几行开始读（从 1 开始）。读大文件时分段使用。"
                        },
                        "limit": {
                            "type": "integer",
                            "description": "最多读多少行。建议值 200，大文件分页用。"
                        }
                    },
                    "required": ["path"]
                }
            }
        }

    def validate(self, params: dict) -> list[str]:
        errors = []
        if "path" not in params:
            errors.append("缺少必填参数: path")
        if "offset" in params and not isinstance(params["offset"], int):
            errors.append("offset 必须是整数")
        if "limit" in params and not isinstance(params["limit"], int):
            errors.append("limit 必须是整数")
        if "offset" in params and params.get("offset", 1) < 1:
            errors.append("offset 最小为 1")
        return errors

    async def execute(self, params: dict) -> ToolResult:
        try:
            path = resolve_path(params["path"], self.cwd)
        except PermissionError as e:
            return ToolResult.error(str(e))

        # 文件存在检查
        if not path.exists():
            return ToolResult.error(f"文件不存在: {path}")
        if not path.is_file():
            return ToolResult.error(f"'{path}' 不是文件（可能是目录）")

        # 大小检查
        size = path.stat().st_size
        if size > MAX_FILE_SIZE:
            return ToolResult.error(
                f"文件过大（{size / 1024 / 1024:.1f} MB），超过限制 10 MB。"
                f"请使用 offset/limit 分段读取。"
            )

        # 二进制检查
        if is_binary(path):
            return ToolResult.error(f"不支持读取二进制文件: {path.name}")

        # 读取文件
        try:
            content = path.read_text(encoding="utf-8", errors="replace")
        except Exception as e:
            return ToolResult.error(f"读取失败: {e}")

        lines = content.splitlines()
        total_lines = len(lines)

        # 分页处理
        offset = params.get("offset", 1)
        limit = params.get("limit", None)

        start = offset - 1  # 转为 0-based 索引
        end = min(start + limit, total_lines) if limit else total_lines
        selected_lines = lines[start:end]

        result_text = add_line_numbers("\n".join(selected_lines), start_line=offset)

        # Token 截断保护
        truncated = False
        if len(result_text) > MAX_READ_CHARS:
            result_text = result_text[:MAX_READ_CHARS]
            result_text += f"\n\n[内容已截断，共 {total_lines} 行，当前显示第 {offset}-{offset+len(selected_lines)-1} 行]"
            truncated = True

        # 如果有更多内容，提示 LLM
        has_more = end < total_lines
        if has_more and not truncated:
            result_text += f"\n\n[文件共 {total_lines} 行，当前显示第 {offset}-{end} 行。使用 offset={end+1} 读取后续内容]"

        return ToolResult.ok(
            result_text,
            path=str(path),
            total_lines=total_lines,
            shown_lines=len(selected_lines),
            truncated=truncated,
        )

In [ ]:
# 测试：读这个 Notebook 所在目录的文件
tool = ReadFileTool(cwd=Path("."))

# 读 src/llm_client.py
result = await tool.execute({"path": "src/llm_client.py"})
print(f"读取结果: success={result.success}")
print(f"元数据: {result.metadata}")
print("\n前 300 字符:")
print(result.content[:300])

In [ ]:
# 测试分页：只读前 5 行
result = await tool.execute({"path": "src/llm_client.py", "offset": 1, "limit": 5})
print("前 5 行:")
print(result.content)
print()

# 测试文件不存在
result = await tool.execute({"path": "nonexistent.py"})
print("不存在的文件:")
print(f"  success={result.success}, content={result.content}")
print()

# 测试路径穿越
result = await tool.execute({"path": "../../etc/passwd"})
print("路径穿越:")
print(f"  success={result.success}, content={result.content[:80]}")

## 3. 工具二：WriteFileTool

写文件比读文件多几个考虑：

- **自动创建目录**：写到 `src/utils/helpers.py`，如果 `src/utils/` 不存在要自动创建
- **生成 diff**：记录写之前的内容，用 `difflib` 生成 unified diff，方便用户 review
- **区分新建和更新**：`action` 字段告诉用户是 created 还是 updated

In [ ]:
import difflib
from dataclasses import dataclass


@dataclass
class FileDiff:
    """记录文件修改前后的差异。"""
    path: str
    old_content: str
    new_content: str
    is_new_file: bool = False

    def to_unified_diff(self, context_lines: int = 3) -> str:
        """生成 unified diff 字符串（就是 git diff 那种格式）。"""
        old_lines = self.old_content.splitlines(keepends=True)
        new_lines = self.new_content.splitlines(keepends=True)

        diff = difflib.unified_diff(
            old_lines,
            new_lines,
            fromfile=f"a/{self.path}" if not self.is_new_file else "/dev/null",
            tofile=f"b/{self.path}",
            n=context_lines,
        )
        return "".join(diff)


# 演示 diff 格式
old = "def hello():\n    print('Hello')\n    return 1\n"
new = "def hello(name: str = 'World'):\n    print(f'Hello, {name}!')\n    return 0\n"

d = FileDiff(path="hello.py", old_content=old, new_content=new)
print(d.to_unified_diff())

In [ ]:
class WriteFileTool(Tool):

    def __init__(self, cwd: Path = Path(".")):
        self.cwd = cwd.resolve()

    @property
    def name(self) -> str:
        return "write_file"

    @property
    def description(self) -> str:
        return (
            "写入或覆盖文件内容。如果文件不存在则创建，父目录不存在则自动创建。"
            "适合写入全新文件或完整替换文件内容。"
            "如果只需要修改文件中的某一段，优先用 edit_file（更精确、diff 更小）。"
        )

    @property
    def kind(self) -> ToolKind:
        return ToolKind.WRITE

    def schema(self) -> dict:
        return {
            "type": "function",
            "function": {
                "name": self.name,
                "description": self.description,
                "parameters": {
                    "type": "object",
                    "properties": {
                        "path": {"type": "string", "description": "文件路径"},
                        "content": {"type": "string", "description": "要写入的完整内容"},
                        "create_directories": {
                            "type": "boolean",
                            "description": "父目录不存在时是否自动创建，默认 true"
                        }
                    },
                    "required": ["path", "content"]
                }
            }
        }

    def validate(self, params: dict) -> list[str]:
        errors = []
        if "path" not in params:
            errors.append("缺少必填参数: path")
        if "content" not in params:
            errors.append("缺少必填参数: content")
        return errors

    async def execute(self, params: dict) -> ToolResult:
        try:
            path = resolve_path(params["path"], self.cwd)
        except PermissionError as e:
            return ToolResult.error(str(e))

        content = params["content"]
        create_dirs = params.get("create_directories", True)

        # 读旧内容（用于生成 diff）
        is_new = not path.exists()
        old_content = ""
        if not is_new:
            try:
                old_content = path.read_text(encoding="utf-8", errors="replace")
            except Exception:
                pass  # 读不到旧内容就跳过 diff

        # 创建父目录
        if create_dirs:
            path.parent.mkdir(parents=True, exist_ok=True)

        # 写入
        try:
            path.write_text(content, encoding="utf-8")
        except Exception as e:
            return ToolResult.error(f"写入失败: {e}")

        # 生成 diff
        diff = FileDiff(
            path=str(path.relative_to(self.cwd)),
            old_content=old_content,
            new_content=content,
            is_new_file=is_new,
        )
        diff_str = diff.to_unified_diff()

        action = "created" if is_new else "updated"
        line_count = len(content.splitlines())

        summary = f"文件已{action}: {path}\n共 {line_count} 行"
        if diff_str:
            summary += f"\n\n{diff_str}"

        return ToolResult.ok(
            summary,
            path=str(path),
            action=action,
            line_count=line_count,
            diff=diff_str,
        )

In [ ]:
import tempfile, os

# 用临时目录测试，不污染当前目录
with tempfile.TemporaryDirectory() as tmpdir:
    tmp = Path(tmpdir)
    tool = WriteFileTool(cwd=tmp)

    # 新建文件
    result = await tool.execute({
        "path": "hello.py",
        "content": "def hello():\n    print('Hello')\n    return 1\n"
    })
    print("新建文件:", result.metadata["action"])
    print(result.content)
    print()

    # 修改文件
    result = await tool.execute({
        "path": "hello.py",
        "content": "def hello(name: str = 'World'):\n    print(f'Hello, {name}!')\n    return 0\n"
    })
    print("修改文件:", result.metadata["action"])
    print(result.content)
    print()

    # 自动创建目录
    result = await tool.execute({
        "path": "src/utils/helpers.py",
        "content": "# helpers\n"
    })
    print("自动创建目录:", result.metadata["action"])
    print("路径:", result.metadata["path"])

## 4. 工具三：EditTool

WriteFileTool 是全量替换。EditTool 是**精确手术**——只替换文件中的某一段字符串。

为什么需要 EditTool？
- 改一行代码，不用让 LLM 重写整个文件
- Token 消耗少（LLM 只需要生成被改的部分）
- 更安全：只改指定内容，不会意外覆盖其他部分

**关键设计**：`old_string` 必须在文件中**精确匹配唯一**，否则报错。这是为了确保替换的是对的那一处。

In [ ]:
class EditTool(Tool):

    def __init__(self, cwd: Path = Path(".")):
        self.cwd = cwd.resolve()

    @property
    def name(self) -> str:
        return "edit_file"

    @property
    def description(self) -> str:
        return (
            "精确替换文件中的某段文字。用 old_string 定位要替换的内容，用 new_string 替换它。"
            "old_string 必须在文件中精确存在，多余空格/换行都会导致匹配失败。"
            "默认只替换第一处匹配（replace_all=false）。"
            "如果需要修改整个文件，用 write_file。"
        )

    @property
    def kind(self) -> ToolKind:
        return ToolKind.WRITE

    def schema(self) -> dict:
        return {
            "type": "function",
            "function": {
                "name": self.name,
                "description": self.description,
                "parameters": {
                    "type": "object",
                    "properties": {
                        "path": {"type": "string", "description": "文件路径"},
                        "old_string": {
                            "type": "string",
                            "description": "要被替换的文字，必须与文件内容精确匹配（包括缩进和换行）"
                        },
                        "new_string": {
                            "type": "string",
                            "description": "替换后的文字"
                        },
                        "replace_all": {
                            "type": "boolean",
                            "description": "替换所有匹配处（默认 false，只替换第一处）"
                        }
                    },
                    "required": ["path", "old_string", "new_string"]
                }
            }
        }

    def validate(self, params: dict) -> list[str]:
        errors = []
        for key in ["path", "old_string", "new_string"]:
            if key not in params:
                errors.append(f"缺少必填参数: {key}")
        return errors

    async def execute(self, params: dict) -> ToolResult:
        try:
            path = resolve_path(params["path"], self.cwd)
        except PermissionError as e:
            return ToolResult.error(str(e))

        if not path.exists():
            return ToolResult.error(f"文件不存在: {path}")
        if is_binary(path):
            return ToolResult.error("不支持编辑二进制文件")

        old_string = params["old_string"]
        new_string = params["new_string"]
        replace_all = params.get("replace_all", False)

        try:
            old_content = path.read_text(encoding="utf-8", errors="replace")
        except Exception as e:
            return ToolResult.error(f"读取失败: {e}")

        # 检查 old_string 是否存在
        count = old_content.count(old_string)
        if count == 0:
            # 给 LLM 有用的错误信息，方便它调整重试
            return ToolResult.error(
                f"在文件中找不到要替换的内容。\n"
                f"请确认 old_string 与文件中的内容完全一致（包括缩进、空格、换行）。\n"
                f"提示：用 read_file 先看一眼文件内容，然后精确复制要替换的部分。",
                path=str(path),
            )

        # 多处匹配但没开 replace_all，给出警告
        if count > 1 and not replace_all:
            return ToolResult.error(
                f"找到 {count} 处匹配，但 replace_all=false。"
                f"请提供更多上下文让 old_string 唯一，或者设置 replace_all=true。",
                match_count=count,
            )

        # 执行替换
        if replace_all:
            new_content = old_content.replace(old_string, new_string)
        else:
            new_content = old_content.replace(old_string, new_string, 1)

        path.write_text(new_content, encoding="utf-8")

        diff = FileDiff(
            path=str(path.relative_to(self.cwd)),
            old_content=old_content,
            new_content=new_content,
        )
        diff_str = diff.to_unified_diff()

        replaced_count = count if replace_all else 1
        summary = f"已替换 {replaced_count} 处。\n\n{diff_str}"

        return ToolResult.ok(
            summary,
            path=str(path),
            replaced_count=replaced_count,
            line_count=len(new_content.splitlines()),
            diff=diff_str,
        )

In [ ]:
with tempfile.TemporaryDirectory() as tmpdir:
    tmp = Path(tmpdir)
    
    # 先创建测试文件
    test_file = tmp / "calc.py"
    test_file.write_text(
        "def add(a, b):\n"
        "    return a + b\n"
        "\n"
        "def subtract(a, b):\n"
        "    return a - b\n"
    )
    
    tool = EditTool(cwd=tmp)
    
    # 精确替换一行
    result = await tool.execute({
        "path": "calc.py",
        "old_string": "def add(a, b):",
        "new_string": "def add(a: int, b: int) -> int:"
    })
    print("替换结果:", result.success)
    print(result.content)
    print()

    # 测试找不到的情况
    result = await tool.execute({
        "path": "calc.py",
        "old_string": "def multiply(a, b):",  # 不存在
        "new_string": "..."
    })
    print("找不到时的错误提示:")
    print(result.content)

## 5. 工具四：ListDirectoryTool

列出目录内容。Agent 通常先用这个工具了解项目结构，再决定读哪些文件。

In [ ]:
class ListDirectoryTool(Tool):

    def __init__(self, cwd: Path = Path(".")):
        self.cwd = cwd.resolve()

    @property
    def name(self) -> str:
        return "list_directory"

    @property
    def description(self) -> str:
        return (
            "列出目录内容。目录名后面带斜杠，文件直接显示名称。"
            "用于了解项目结构，决定接下来要读哪些文件。"
        )

    @property
    def kind(self) -> ToolKind:
        return ToolKind.READ

    def schema(self) -> dict:
        return {
            "type": "function",
            "function": {
                "name": self.name,
                "description": self.description,
                "parameters": {
                    "type": "object",
                    "properties": {
                        "path": {
                            "type": "string",
                            "description": "要列出的目录路径，默认为工作目录"
                        },
                        "include_hidden": {
                            "type": "boolean",
                            "description": "是否包含隐藏文件（. 开头），默认 false"
                        }
                    },
                    "required": []
                }
            }
        }

    def validate(self, params: dict) -> list[str]:
        return []

    async def execute(self, params: dict) -> ToolResult:
        path_str = params.get("path", ".")
        include_hidden = params.get("include_hidden", False)

        try:
            path = resolve_path(path_str, self.cwd)
        except PermissionError as e:
            return ToolResult.error(str(e))

        if not path.exists():
            return ToolResult.error(f"路径不存在: {path}")
        if not path.is_dir():
            return ToolResult.error(f"'{path}' 不是目录")

        try:
            entries = sorted(path.iterdir(), key=lambda p: (p.is_file(), p.name.lower()))
        except PermissionError:
            return ToolResult.error(f"没有权限读取目录: {path}")

        lines = []
        for entry in entries:
            if not include_hidden and entry.name.startswith("."):
                continue
            # 目录名加斜杠，一眼能区分
            name = entry.name + "/" if entry.is_dir() else entry.name
            lines.append(name)

        content = "\n".join(lines) if lines else "（空目录）"
        return ToolResult.ok(
            content,
            path=str(path),
            entry_count=len(lines),
        )


# 测试
tool = ListDirectoryTool(cwd=Path("."))
result = await tool.execute({"path": "."})
print("当前目录内容:")
print(result.content)
print(f"\n共 {result.metadata['entry_count']} 条")

## 6. 工具五：GlobTool

按模式批量找文件。比 `list_directory` 更强大——可以递归搜索，可以按扩展名过滤。

常见用法：
- `**/*.py` — 找所有 Python 文件
- `src/**/*.ts` — 找 src 目录下所有 TypeScript 文件
- `**/test_*.py` — 找所有测试文件
- `**/*.{json,yaml}` — 找配置文件（注意：Python 的 glob 不支持 `{}`，需要多次调用）

In [ ]:
MAX_GLOB_RESULTS = 500  # 最多返回 500 个结果


class GlobTool(Tool):

    def __init__(self, cwd: Path = Path(".")):
        self.cwd = cwd.resolve()

    @property
    def name(self) -> str:
        return "glob"

    @property
    def description(self) -> str:
        return (
            "按 glob 模式搜索文件。支持 * (任意字符)、** (递归任意目录)、? (单字符)。"
            "示例：'**/*.py' 找所有 Python 文件，'src/**/*.ts' 找 src 下所有 TS 文件。"
            "只返回文件路径，不包含目录。"
        )

    @property
    def kind(self) -> ToolKind:
        return ToolKind.READ

    def schema(self) -> dict:
        return {
            "type": "function",
            "function": {
                "name": self.name,
                "description": self.description,
                "parameters": {
                    "type": "object",
                    "properties": {
                        "pattern": {
                            "type": "string",
                            "description": "glob 模式，如 '**/*.py' 或 'src/*.ts'"
                        },
                        "search_path": {
                            "type": "string",
                            "description": "搜索的根目录，默认为工作目录"
                        }
                    },
                    "required": ["pattern"]
                }
            }
        }

    def validate(self, params: dict) -> list[str]:
        if "pattern" not in params:
            return ["缺少必填参数: pattern"]
        return []

    async def execute(self, params: dict) -> ToolResult:
        pattern = params["pattern"]
        search_path_str = params.get("search_path", ".")

        try:
            search_path = resolve_path(search_path_str, self.cwd)
        except PermissionError as e:
            return ToolResult.error(str(e))

        if not search_path.is_dir():
            return ToolResult.error(f"搜索路径不是目录: {search_path}")

        try:
            matches = [p for p in search_path.glob(pattern) if p.is_file()]
        except Exception as e:
            return ToolResult.error(f"glob 执行失败: {e}，请检查模式语法")

        # 按路径排序
        matches.sort()

        truncated = False
        if len(matches) > MAX_GLOB_RESULTS:
            matches = matches[:MAX_GLOB_RESULTS]
            truncated = True

        # 返回相对于搜索根目录的路径
        rel_paths = [str(p.relative_to(search_path)) for p in matches]
        content = "\n".join(rel_paths) if rel_paths else "没有匹配的文件"

        if truncated:
            content += f"\n\n[结果已截断，只显示前 {MAX_GLOB_RESULTS} 个。缩小搜索范围或使用更精确的模式。]"

        return ToolResult.ok(
            content,
            match_count=len(rel_paths),
            search_path=str(search_path),
            truncated=truncated,
        )


# 测试
tool = GlobTool(cwd=Path("."))

result = await tool.execute({"pattern": "**/*.py"})
print(f"找到 {result.metadata['match_count']} 个 .py 文件:")
print(result.content)
print()

result = await tool.execute({"pattern": "src/*.py"})
print(f"src/ 下的 .py 文件:")
print(result.content)

## 7. 全部注册，验证 schema 输出

In [ ]:
cwd = Path(".")
registry = ToolRegistry()

registry.register(ReadFileTool(cwd))
registry.register(WriteFileTool(cwd))
registry.register(EditTool(cwd))
registry.register(ListDirectoryTool(cwd))
registry.register(GlobTool(cwd))

print(registry)
print()

# 每个工具的 schema 名称和描述第一句
for schema in registry.get_schemas():
    fn = schema["function"]
    desc_first_line = fn["description"].split("。")[0]
    required = fn["parameters"].get("required", [])
    print(f"  {fn['name']:20s} | 必填: {required}")
    print(f"  {' '*20}   {desc_first_line}")
    print()

## 8. 小结

| 工具 | 关键实现细节 |
|------|-------------|
| `resolve_path` | 展开 `..`，拒绝工作目录外的路径 |
| `ReadFileTool` | 带行号、分页（offset/limit）、大文件限制、二进制检测 |
| `WriteFileTool` | 自动建目录、记录旧内容生成 diff、区分 created/updated |
| `EditTool` | 精确字符串替换、重复匹配保护、找不到时给 LLM 有用的错误提示 |
| `ListDirectoryTool` | 目录名加斜杠、过滤隐藏文件、按名称排序 |
| `GlobTool` | `Path.glob`、只返回文件、500 条结果截断 |

**EditTool 的错误信息设计**值得注意：当 `old_string` 找不到时，不是简单报错，而是告诉 LLM「用 read_file 先看一眼再复制」。这类错误提示直接影响 Agent 能不能自行恢复，写得越清楚，LLM 重试成功率越高。

**下一章：Network Tools**
ReadFile / WriteFile 处理本地文件。下章做 WebSearchTool 和 WebFetchTool，让 Agent 能搜索网络、抓取文档。

---
## 附：保存到 src/

In [ ]:
import shutil

# 把完整实现写成一个文件
file_tools_code = open("src/tool_framework.py").read()  # 需要 ToolKind/ToolResult/Tool/ToolRegistry

# 实际项目里应该分开文件，这里为了简单合到一起
# src/file_tools.py
import_code = '''
# file_tools.py — 文件系统工具
# 使用: from src.file_tools import ReadFileTool, WriteFileTool, EditTool, ListDirectoryTool, GlobTool
'''

print("完整代码太长，已在 Notebook 单元格中定义。")
print("下一章从 Notebook 直接 import 这些工具类。")
print()
print("快速验证所有工具可用:")
for t in [ReadFileTool, WriteFileTool, EditTool, ListDirectoryTool, GlobTool]:
    inst = t()
    print(f"  ✓ {inst.name:20s} kind={inst.kind.value}, mutating={inst.is_mutating()}")